##### Copyright 2026 Google LLC.

In [ ]:
# @title Licensed under the Apache License, Version 2.0
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# PostHog LLM Analytics with the Gemini API

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/examples/posthog/PostHog_LLM_Analytics_with_Gemini.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/>
</a>

## Overview

This notebook shows you how to combine [PostHog](https://posthog.com) product analytics with the Gemini API to build AI-powered insights on top of your event data.

You will learn how to:

1. **Query PostHog events with HogQL** — pull raw usage data from your PostHog project
2. **Summarize LLM analytics** — analyze AI feature costs, latency, and token usage
3. **Evaluate feature flag impact** — use Gemini to compare conversion metrics between flag variants
4. **Generate a plain-language analytics report** — turn a dashboard snapshot into a stakeholder-ready narrative

### Prerequisites

- A Gemini API key in the environment variable `GEMINI_API_KEY`
- A PostHog personal API key in `POSTHOG_API_KEY` ([create one here](https://app.posthog.com/settings/user-api-keys))
- Your PostHog project ID in `POSTHOG_PROJECT_ID` (found in **Project Settings → General**)
- PostHog host in `POSTHOG_HOST` (default: `https://us.i.posthog.com`; EU cloud: `https://eu.i.posthog.com`)

## Setup

In [ ]:
%pip install -U -q 'google-genai>=1.0.0' requests

In [ ]:
import json
import os

import requests
from google import genai
from google.genai import types
from IPython.display import Markdown, display

### Configure API clients

Store your keys as environment variables or, in Colab, as Colab Secrets.

In [ ]:
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

POSTHOG_API_KEY = os.environ["POSTHOG_API_KEY"]
POSTHOG_PROJECT_ID = os.environ["POSTHOG_PROJECT_ID"]
POSTHOG_HOST = os.environ.get("POSTHOG_HOST", "https://us.i.posthog.com")

posthog_headers = {
    "Authorization": f"Bearer {POSTHOG_API_KEY}",
    "Content-Type": "application/json",
}

In [ ]:
MODEL_ID = "gemini-2.5-flash"  # @param ["gemini-2.5-flash-lite", "gemini-2.5-flash", "gemini-2.5-pro", "gemini-3-flash-preview"] {"allow-input":true, isTemplate: true}

## 1. Query PostHog events with HogQL

[HogQL](https://posthog.com/docs/hogql) is PostHog's SQL dialect. The Query API lets you run HogQL queries programmatically and get results as JSON — which you can then pass directly to Gemini.

In [ ]:
hogql_query = """
SELECT
    event,
    count() AS total,
    countIf(toDate(timestamp) = today()) AS today
FROM events
WHERE timestamp >= now() - INTERVAL 7 DAY
GROUP BY event
ORDER BY total DESC
LIMIT 20
"""

query_response = requests.post(
    f"{POSTHOG_HOST}/api/projects/{POSTHOG_PROJECT_ID}/query",
    headers=posthog_headers,
    json={"query": {"kind": "HogQLQuery", "query": hogql_query}},
)
query_response.raise_for_status()
event_data = query_response.json()

print(json.dumps(event_data, indent=4))

Ask Gemini to summarize the most important signals in the event data.

In [ ]:
event_summary_response = client.models.generate_content(
    model=MODEL_ID,
    contents=f"""
You are a product analyst. The following JSON contains event counts from a
PostHog project over the last 7 days. Summarize the three most important
insights in plain language. Call out any events that look anomalous.

Event data:
{json.dumps(event_data, indent=2)}
""",
)

display(Markdown(event_summary_response.text))

## 2. Summarize LLM analytics

If you use PostHog's [LLM observability](https://posthog.com/docs/ai-engineering/llm-observability) to capture `$ai_generation` events, you can query token usage, latency, and costs and ask Gemini to surface the most actionable findings.

In [ ]:
llm_query = """
SELECT
    properties.$ai_model AS model,
    count() AS requests,
    avg(properties.$ai_latency) AS avg_latency_ms,
    sum(properties.$ai_input_tokens) AS total_input_tokens,
    sum(properties.$ai_output_tokens) AS total_output_tokens,
    sum(properties.$ai_total_cost_usd) AS total_cost_usd
FROM events
WHERE event = '$ai_generation'
  AND timestamp >= now() - INTERVAL 7 DAY
GROUP BY model
ORDER BY total_cost_usd DESC
"""

llm_response = requests.post(
    f"{POSTHOG_HOST}/api/projects/{POSTHOG_PROJECT_ID}/query",
    headers=posthog_headers,
    json={"query": {"kind": "HogQLQuery", "query": llm_query}},
)
llm_response.raise_for_status()
llm_data = llm_response.json()

print(json.dumps(llm_data, indent=4))

Use Gemini with structured output to extract a cost and performance breakdown per model.

In [ ]:
llm_schema = types.Schema(
    type=types.Type.OBJECT,
    properties={
        "total_cost_usd": types.Schema(type=types.Type.NUMBER),
        "most_expensive_model": types.Schema(type=types.Type.STRING),
        "highest_latency_model": types.Schema(type=types.Type.STRING),
        "recommendations": types.Schema(
            type=types.Type.ARRAY,
            items=types.Schema(type=types.Type.STRING),
        ),
    },
    required=["total_cost_usd", "most_expensive_model",
              "highest_latency_model", "recommendations"],
)

llm_analysis_response = client.models.generate_content(
    model=MODEL_ID,
    contents=f"""
Analyze the following LLM usage data from PostHog and return a structured report.
Identify cost drivers and latency outliers. Provide 2-3 specific recommendations
to reduce cost or improve performance.

LLM usage data:
{json.dumps(llm_data, indent=2)}
""",
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=llm_schema,
    ),
)

llm_report = json.loads(llm_analysis_response.text)
print(json.dumps(llm_report, indent=4))

## 3. Evaluate feature flag impact

Fetch conversion metrics split by feature flag variant and ask Gemini to interpret the results — answering whether the flag rollout had a statistically meaningful impact.

In [ ]:
FLAG_KEY = "new-onboarding"  # @param {type: "string"}

flag_query = f"""
SELECT
    person.properties.$feature_flag_response.`{FLAG_KEY}` AS variant,
    countIf(event = 'signup_completed') AS signups,
    countIf(event = 'pageview') AS pageviews,
    round(
        100.0 * countIf(event = 'signup_completed')
        / nullIf(countIf(event = 'pageview'), 0),
        2
    ) AS conversion_pct
FROM events
WHERE timestamp >= now() - INTERVAL 14 DAY
  AND variant IS NOT NULL
GROUP BY variant
ORDER BY variant
"""

flag_response = requests.post(
    f"{POSTHOG_HOST}/api/projects/{POSTHOG_PROJECT_ID}/query",
    headers=posthog_headers,
    json={"query": {"kind": "HogQLQuery", "query": flag_query}},
)
flag_response.raise_for_status()
flag_data = flag_response.json()

print(json.dumps(flag_data, indent=4))

In [ ]:
flag_interpretation_response = client.models.generate_content(
    model=MODEL_ID,
    contents=f"""
You are an A/B test analyst. The following data shows conversion rates
for each variant of the '{FLAG_KEY}' feature flag over 14 days.

Interpret the results:
1. Which variant performed better and by how much?
2. Is the difference likely meaningful given the sample sizes shown?
3. What would you recommend — roll out, roll back, or gather more data?

Flag data:
{json.dumps(flag_data, indent=2)}
""",
)

display(Markdown(flag_interpretation_response.text))

## 4. Generate a plain-language analytics report

Combine all three data sources into a single executive summary — the kind you'd paste into a weekly product update or Slack message.

In [ ]:
report_response = client.models.generate_content(
    model=MODEL_ID,
    contents=f"""
You are a product analyst writing a weekly report for a non-technical audience.
Use the three data sources below to write a 3-5 paragraph summary covering:
- Top product activity highlights from the last 7 days
- AI/LLM cost and usage summary with key recommendations
- Feature flag experiment results and next-step recommendation

Keep the language plain. Use specific numbers. End with one clear call to action.

Event summary data:
{json.dumps(event_data, indent=2)}

LLM analytics data:
{json.dumps(llm_data, indent=2)}

Feature flag data ({FLAG_KEY}):
{json.dumps(flag_data, indent=2)}
""",
)

display(Markdown(report_response.text))

## Next steps

You've seen four PostHog + Gemini integration patterns:

1. **HogQL event queries** — pull any event data into Gemini for analysis
2. **LLM cost analytics** — surface cost drivers and latency outliers automatically
3. **Feature flag evaluation** — interpret A/B test results in plain language
4. **Executive reporting** — synthesize multiple data sources into one narrative

### Continue exploring

- Use **streaming** to display reports incrementally — see [`quickstarts/Streaming.ipynb`](../../quickstarts/Streaming.ipynb)
- Add **function calling** so Gemini can query PostHog autonomously — see [`quickstarts/Function_calling.ipynb`](../../quickstarts/Function_calling.ipynb)
- Track your Gemini API calls back in PostHog with the [PostHog LLM observability SDK](https://posthog.com/docs/ai-engineering/llm-observability)
- Explore the full PostHog Query API: [posthog.com/docs/api/query](https://posthog.com/docs/api/query)